In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import torch
from segmentation_models_pytorch import Unet
import kornia.augmentation as K
from torchmetrics.segmentation import MeanIoU, DiceScore
import lightning as L
from torchinfo import summary

from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from datamodules.datamodule_deadtrees import deadtrees_dm
from segmentators import RegularTrainingSegmentator
from segmentation_models_pytorch.losses import DiceLoss

VERSION = 2
CHECKPOINT_PATH = (
    f"checkpoints/deadtrees_segmentation/version_{VERSION}/checkpoints/last.ckpt"
)
PROJECT_NAME = "deadtrees_segmentation"

In [3]:
model = Unet("resnet34", in_channels=3, classes=2)
augmentation = dict(
    geom_augmentation=K.AugmentationSequential(
        K.RandomHorizontalFlip(p=0.5),
        K.RandomVerticalFlip(p=0.5),
        data_keys=["input", "mask"],
    ),
    intensity_augmentation=K.AugmentationSequential(
        K.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.8),
        K.RandomGaussianNoise(mean=0.0, std=0.05, p=0.3),
        data_keys=["input"],  # solo imagen
    ),
)


segmentator = RegularTrainingSegmentator(
    model=model,
    criterion=DiceLoss(mode="binary"),
    augmentation=None,
    optimizer_cls=torch.optim.AdamW,
    optimizer_kwargs=dict(lr=1e-4, weight_decay=1e-4),
    metric=DiceScore(
        num_classes=2,
        include_background=False,
    ),
    add_channel_dim=True,
)


In [4]:
summary(segmentator, input_size=(1, 3, 256, 256), device="cpu")

Layer (type:depth-idx)                             Output Shape              Param #
RegularTrainingSegmentator                         [1, 2, 256, 256]          --
├─Unet: 1-1                                        [1, 2, 256, 256]          --
│    └─ResNetEncoder: 2-1                          [1, 3, 256, 256]          --
│    │    └─Conv2d: 3-1                            [1, 64, 128, 128]         9,408
│    │    └─BatchNorm2d: 3-2                       [1, 64, 128, 128]         128
│    │    └─ReLU: 3-3                              [1, 64, 128, 128]         --
│    │    └─MaxPool2d: 3-4                         [1, 64, 64, 64]           --
│    │    └─Sequential: 3-5                        [1, 64, 64, 64]           221,952
│    │    └─Sequential: 3-6                        [1, 128, 32, 32]          1,116,416
│    │    └─Sequential: 3-7                        [1, 256, 16, 16]          6,822,400
│    │    └─Sequential: 3-8                        [1, 512, 8, 8]            13,114,368
│   

In [5]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f"checkpoints/deadtrees_segmentation/version_{VERSION}/checkpoints",
    filename="{epoch:02d}-{step}-{val_dice:.4f}-{val_loss:.4f}",
    save_top_k=1,  # keep best 3 models
    monitor="val_dice",  # or "val_loss"
    mode="max",  # IoU → maximize
    save_last=True,  # VERY IMPORTANT for resume
)
logger = TensorBoardLogger(save_dir="logs", version=VERSION, name="minimal_example")
trainer = L.Trainer(
    max_epochs=30,
    logger=logger,
    callbacks=[checkpoint_callback],
    accelerator="gpu",
    devices=1,
    enable_progress_bar=True,
    deterministic=False,  ## Not possible for CE
    check_val_every_n_epoch=1,
    fast_dev_run=False,  # Para probar que el código corre sin errores, luego quitarlo para entrenar en serio
    overfit_batches=0,
    log_every_n_steps=1,
    precision="16-mixed",
    accumulate_grad_batches=4,
)

# trainer.fit(segmentator, deadtrees_dm, ckpt_path=CHECKPOINT_PATH)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [6]:
deadtrees_dm.setup()
loader = deadtrees_dm.val_dataloader()
for batch in loader:
    print(batch["image"].shape, batch["mask"].shape)
    break

torch.Size([16, 3, 256, 256]) torch.Size([16, 1, 256, 256])


In [7]:
segmentator.validation_step(batch, 0)

/home/datacuber/Documents/semantic_segmentation/.venv/lib/python3.13/site-packages/lightning/pytorch/core/module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


In [9]:
batch["image"][0]

tensor([[[ 90.,  65.,  92.,  ..., 100., 111., 126.],
         [ 75.,  99.,  57.,  ...,  93.,  91., 131.],
         [109.,  93.,  78.,  ..., 128., 141., 128.],
         ...,
         [139.,  88.,  92.,  ...,  90., 108.,  73.],
         [ 96., 107.,  66.,  ...,  97.,  98.,  80.],
         [130., 115., 114.,  ..., 110., 117.,  96.]],

        [[111.,  90., 112.,  ...,  92., 100., 113.],
         [ 96., 124.,  76.,  ...,  85.,  86., 125.],
         [130., 116.,  98.,  ..., 120., 134., 119.],
         ...,
         [131.,  77.,  81.,  ...,  83.,  97.,  61.],
         [ 90.,  97.,  57.,  ...,  91.,  88.,  67.],
         [122., 102., 104.,  ..., 106., 111.,  87.]],

        [[ 60.,  36.,  70.,  ...,  77.,  90.,  98.],
         [ 48.,  73.,  37.,  ...,  69.,  71., 109.],
         [ 83.,  66.,  52.,  ..., 102., 111., 103.],
         ...,
         [109.,  59.,  62.,  ...,  65.,  82.,  42.],
         [ 64.,  80.,  41.,  ...,  75.,  69.,  52.],
         [ 96.,  82.,  89.,  ...,  89.,  92.,  65.]]]